# Validating the LLM judge's corrections

The DPO pairs' `chosen` side for the **aborted** (AB) and **failed** (F) conditions is a bare
corrected move written by the judge (GPT-5.2); the reflection text that motivated it was
discarded before training. This notebook asks: **were those suggested moves any good?** —
validated against *game ground truth*, not opinion.

Why it matters: the F pairs damaged performance at both scales. Two competing explanations:
1. the judge's moves were simply bad (content problem), or
2. the moves were fine but off-policy — text the model doesn't generate — and raising their
   likelihood corroded formatting (distribution problem, the elicitation story).
Validating correction quality decides between them. Bonus contrast: the AB pairs *helped* —
if AB and F corrections have similar validity, content quality cannot explain the opposite
outcomes.

**Tiers** (by strength of ground truth):
- **Tier 1 — objectively verifiable**: wordle (feedback consistency), codenames (board rules).
- **Tier 2 — legality**: textmapworld family (suggested direction must exist in the current
  room, parsed from the pair's own prompt).
- **Tier 3 — format**: privateshared / guesswhat / adventuregame (does the correction conform
  to the game's required tag format — the thing the F damage allegedly broke).
- **Tier 4 — cross-judge** (not run here): a second LLM judge for the semantic remainder.

Every validator returns `(verdict, reason)` with verdict in `{valid, invalid, unverifiable}`;
unverifiable pairs are reported, never silently dropped. All state is parsed from the pair's own
`prompt` (the conversation up to the correction point), so no transcript linkage is needed.

In [1]:
import json, re
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from IPython.display import display

REPO = Path.cwd()
if not (REPO / "data").exists():
    for p in REPO.parents:
        if (p / "data" / "dpo_pairs").exists():
            REPO = p; break
P = REPO / "data" / "dpo_pairs"

SETS = {
    ("9B", "failed"):  ["failed-pairs-hidden-states.Qwen3.5-9B-sft.json",
                        "failed-pairs-no-hidden-states.Qwen3.5-9B-sft.json"],
    ("9B", "aborted"): ["aborted-rounds-pairs.Qwen3.5-9B-sft.json"],
    ("2B", "failed"):  ["failed-pairs.Qwen3.5-2B-sft-full-merged.gpt-5.2.json"],
    ("2B", "aborted"): ["aborted-rounds-pairs.Qwen3.5-2B-sft.json"],
}

pairs = []
for (size, pairset), files in SETS.items():
    for fn in files:
        fp = P / fn
        if not fp.exists():
            print("MISSING (skipped):", fn); continue
        for p in json.load(open(fp)):
            pairs.append(dict(size=size, pairset=pairset, game=p["game"],
                              pair_id=p.get("pair_id"), prompt=p["prompt"],
                              chosen=p["chosen"][0]["content"],
                              rejected=p["rejected"][0]["content"]))
DF = pd.DataFrame(pairs)
display(DF.groupby(["size", "pairset"]).game.value_counts().unstack(fill_value=0))

game          adventuregame  codenames  guesswhat  imagegame  matchit_ascii  \
size pairset                                                                  
2B   aborted              0         50          0          0              0   
     failed              35         58         16         30             14   
9B   aborted              0         24          0          0              0   
     failed              30         69         27          7              0   

game          privateshared  referencegame  taboo  textmapworld  \
size pairset                                                      
2B   aborted              0              0      0             0   
     failed              45             29      7            45   
9B   aborted              0              0      0             0   
     failed              45              9      5            45   

game          textmapworld_graphreasoning  textmapworld_specificroom  wordle  \
size pairset                                                                   
2B   aborted                            0                          0      16   
     failed                            27                         14       2   
9B   aborted                            0                          0       9   
     failed                            27                         14       2   

game          wordle_withclue  wordle_withcritic  
size pairset                                      
2B   aborted                6                  7  
     failed                 1                  2  
9B   aborted                6                  3  
     failed                 2                  4

In [2]:
# --- shared helpers -------------------------------------------------------
def prompt_text(p, role=None):
    # Concatenated prompt message contents (optionally one role only).
    return "\n".join(str(m["content"]) for m in p["prompt"]
                     if role is None or m["role"] == role)

def last_user_msg(p):
    us = [str(m["content"]) for m in p["prompt"] if m["role"] == "user"]
    return us[-1] if us else ""

def verdict_row(p, verdict, reason, tier):
    return dict(size=p["size"], pairset=p["pairset"], game=p["game"],
                verdict=verdict, reason=reason, tier=tier,
                chosen=p["chosen"][:80], rejected=p["rejected"][:80])

RESULTS = []   # every validator appends verdict_row dicts here

## Tier 1a — wordle: is the suggested guess consistent with the feedback so far?

The pair's prompt contains every `guess_feedback:` line revealed before the correction point.
A *valid* correction proposes a 5-letter word that (i) keeps all known green letters in place,
(ii) avoids letters known absent (duplicate-safe), and (iii) doesn't replay a yellow letter in
the same position. The same test is run on the rejected (actually played) move for contrast.

In [3]:
FB_LINE = re.compile(r"guess_feedback:\s*((?:[a-z]<(?:green|yellow|red)>\s*)+)")
FB_TOK  = re.compile(r"([a-z])<(green|yellow|red)>")
GUESS   = re.compile(r"guess:\s*([a-z]+)", re.I)

def wordle_state(p):
    # Skip prompt msg 0: the game instructions contain a worked example
    # ("guess: apple" + feedback) that must not enter the knowledge state.
    hist = "\n".join(str(m["content"]) for m in p["prompt"][1:])
    greens, present, absent, yellow_at = {}, set(), set(), set()
    for m in FB_LINE.finditer(hist):
        fb = FB_TOK.findall(m.group(1))
        for pos, (ch, col) in enumerate(fb):
            if col == "green":
                greens[pos] = ch; present.add(ch); absent.discard(ch)
            elif col == "yellow":
                present.add(ch); absent.discard(ch); yellow_at.add((ch, pos))
            elif col == "red" and ch not in present:
                absent.add(ch)
    return greens, absent, yellow_at

def check_wordle_move(text, greens, absent, yellow_at):
    m = GUESS.search(text)
    if not m:
        if re.search(r"\b(agree|disagree)\b", text, re.I):
            return "unverifiable", "critic-role move (no guess to validate)"
        return "invalid", "no guess: tag"
    g = m.group(1).lower()
    if len(g) != 5 or not g.isalpha():
        return "invalid", f"not a 5-letter word: {g!r}"
    probs = []
    if any(g[pos] != ch for pos, ch in greens.items()):
        probs.append("drops green")
    dead = sorted(set(g) & absent)
    if dead:
        probs.append(f"uses dead letters {dead}")
    if any((ch, pos) in yellow_at for pos, ch in enumerate(g)):
        probs.append("yellow same pos")
    return ("invalid", "; ".join(probs)) if probs else ("valid", "feedback-consistent")

wsub = DF[DF.game.str.startswith("wordle")]
wcontrast = []
for _, p in wsub.iterrows():
    st = wordle_state(p)
    v, r = check_wordle_move(p["chosen"], *st)
    RESULTS.append(verdict_row(p, v, r, "T1-wordle"))
    vr, rr = check_wordle_move(p["rejected"], *st)
    wcontrast.append(dict(size=p["size"], pairset=p["pairset"], game=p["game"],
                          chosen=v, chosen_why=r, rejected=vr, rejected_why=rr))
WC = pd.DataFrame(wcontrast)
print(f"{len(WC)} wordle pairs")
display(WC.groupby(["pairset"])[["chosen", "rejected"]]
          .agg(lambda s: f"{(s == 'valid').mean():.0%} valid"))
display(WC)

60 wordle pairs


,chosen,rejected
pairset,,
aborted,74% valid,2% valid
failed,77% valid,46% valid


,size,pairset,game,chosen,chosen_why,rejected,rejected_why
0,9B,failed,wordle_withcritic,unverifiable,critic-role move (no guess to validate),invalid,no guess: tag
1,9B,failed,wordle,valid,feedback-consistent,invalid,yellow same pos
2,9B,failed,wordle_withcritic,invalid,no guess: tag,invalid,no guess: tag
3,9B,failed,wordle,valid,feedback-consistent,invalid,uses dead letters ['a']
4,9B,failed,wordle_withcritic,valid,feedback-consistent,valid,feedback-consistent
5,9B,failed,wordle_withcritic,valid,feedback-consistent,valid,feedback-consistent
6,9B,failed,wordle_withclue,valid,feedback-consistent,valid,feedback-consistent
7,9B,failed,wordle_withclue,valid,feedback-consistent,valid,feedback-consistent
8,9B,aborted,wordle,valid,feedback-consistent,invalid,not a 5-letter word: 'stable'
9,9B,aborted,wordle,invalid,drops green; yellow same pos,invalid,no guess: tag


## Tier 1b — codenames: board-rule compliance

From the pair's own prompt: the clue-giver's team words and the full board are listed. Rules
checkable objectively: the CLUE must be a single word **not on the board**; every TARGET must be
one of the team's own words. (Strategic quality of the clue is Tier-4 territory; rule compliance
is exactly the failure mode the damage analysis points at.)

In [4]:
CLUE_RE   = re.compile(r"CLUE:\s*([A-Za-z ]+?)\s*(?:\||$|\n)")
TARGET_RE = re.compile(r"TARGETS?:\s*([^\n|]+)")
GUESS_RE  = re.compile(r"GUESS:\s*([^\n|]+)")

LIST_RES = {
    "team":       re.compile(r"Your team words are:\s*([^\n]+)", re.I),
    "opponent":   re.compile(r"opponent'?s team words are:\s*([^\n]+)", re.I),
    "distractor": re.compile(r"Distractor words are:\s*([^\n]+)", re.I),
}

def codenames_lists(p):
    # Parse the explicit team/opponent/distractor lists; board = their union.
    txt = prompt_text(p)
    found = {}
    for k, rx in LIST_RES.items():
        m = rx.search(txt)
        if m:
            found[k] = set(w.strip(" .").lower()
                           for w in re.split(r"[,;]", m.group(1)) if w.strip(" ."))
    team = found.get("team")
    board = set().union(*found.values()) if found else None
    return team, board

def check_codenames(p):
    team, board = codenames_lists(p)
    c = p["chosen"]
    if GUESS_RE.search(c) and not CLUE_RE.search(c):     # guesser role
        gs = [w.strip().lower() for w in re.split(r"[,;]", GUESS_RE.search(c).group(1)) if w.strip()]
        lu = last_user_msg(p)
        # candidate list: the final comma-separated block of the guesser prompt
        block = lu.split("\n\n")[-1]
        cands = {w.strip().lower() for w in block.split(",") if w.strip()}
        if len(cands) < 5:                     # no plausible list found
            if board is None:
                return "unverifiable", "no word lists in prompt (guesser role)"
            cands = board
        probs = []
        bad = [g for g in gs if g not in cands]
        if bad:
            probs.append(f"guess not among candidates: {bad}")
        mmax = re.search(r"up to (\d+)", lu)
        if mmax and len(gs) > int(mmax.group(1)):
            probs.append(f"{len(gs)} guesses exceeds allowed {mmax.group(1)}")
        mclue = re.search(r"associated with the word '([^']+)'", lu)
        if mclue and mclue.group(1).strip().lower() in gs:
            probs.append("guessed the clue word itself")
        return ("invalid", "; ".join(probs)) if probs else ("valid", "guesses on candidate list")
    mc, mt = CLUE_RE.search(c), TARGET_RE.search(c)
    if not mc:
        return "invalid", "no CLUE tag"
    clue = mc.group(1).strip().lower()
    probs = []
    if len(clue.split()) != 1:
        probs.append("clue not a single word")
    if board and clue in board:
        probs.append(f"clue {clue!r} appears on board")
    if mt and team:
        tg = [w.strip().lower() for w in re.split(r"[,;]", mt.group(1)) if w.strip()]
        bad = [t for t in tg if t not in team]
        if bad:
            probs.append(f"targets not own team words: {bad}")
    elif mt and team is None:
        return "unverifiable", "team list not parsed from prompt"
    return ("invalid", "; ".join(probs)) if probs else ("valid", "rule-compliant")

for _, p in DF[DF.game == "codenames"].iterrows():
    v, r = check_codenames(p)
    RESULTS.append(verdict_row(p, v, r, "T1-codenames"))
cn = pd.DataFrame([x for x in RESULTS if x["tier"] == "T1-codenames"])
display(cn.groupby(["size", "pairset"]).verdict.value_counts().unstack(fill_value=0))
print("\ninvalid examples:")
display(cn[cn.verdict == "invalid"].head(8)[["size", "pairset", "reason", "chosen"]])

verdict       valid
size pairset       
2B   aborted     50
     failed      58
9B   aborted     24
     failed      69


invalid examples:


,size,pairset,reason,chosen


## Tier 2 — textmapworld family: is the suggested direction legal?

The last GM message before the correction states the current room's exits
("Currently available directions: ..."). A `GO: <dir>` correction is *legal* iff the direction
is in that list. Corrections that instead say `DONE` are classified separately (they claim all
rooms are visited — not verifiable from the prompt alone). This is the family where the F pairs
were harvested and where they did the most damage, so an illegal-move rate here would directly
support the content-problem explanation.

In [5]:
DIRS_RE = re.compile(r"currently available directions:\s*([^\n.!?]+)", re.I)
GO_RE   = re.compile(r"GO:\s*([a-z]+)", re.I)

def check_tmw(p):
    lu = last_user_msg(p)
    m = DIRS_RE.search(lu) or DIRS_RE.search(prompt_text(p))
    c = p["chosen"].strip()
    if re.search(r"\bDONE\b", c) and not GO_RE.search(c):
        return "unverifiable", "DONE (needs full map to verify)"
    g = GO_RE.search(c)
    if not g:
        return "invalid", "no GO/DONE tag"
    if not m:
        return "unverifiable", "no direction list in prompt"
    dirs = {d.strip().lower() for d in re.split(r"[,;]|and", m.group(1)) if d.strip()}
    d = g.group(1).lower()
    return ("valid", f"{d} in {sorted(dirs)}") if d in dirs \
           else ("invalid", f"{d} NOT in {sorted(dirs)}")

tmw_games = [g for g in DF.game.unique() if g.startswith("textmapworld")]
for _, p in DF[DF.game.isin(tmw_games)].iterrows():
    v, r = check_tmw(p)
    RESULTS.append(verdict_row(p, v, r, "T2-textmapworld"))
tm = pd.DataFrame([x for x in RESULTS if x["tier"] == "T2-textmapworld"])
display(tm.groupby(["size", "pairset", "game"]).verdict.value_counts().unstack(fill_value=0))
print("\nillegal-direction examples:")
display(tm[tm.verdict == "invalid"].head(8)[["size", "pairset", "game", "reason", "chosen"]])

verdict                                   invalid  valid
size pairset game                                       
2B   failed  textmapworld                       1     44
             textmapworld_graphreasoning        0     27
             textmapworld_specificroom          0     14
9B   failed  textmapworld                       0     45
             textmapworld_graphreasoning        0     27
             textmapworld_specificroom          0     14


illegal-direction examples:


,size,pairset,game,reason,chosen
125,2B,failed,textmapworld,"south NOT in ['east', 'west']",GO: south


## Tier 3 — format conformance: adventuregame, privateshared, guesswhat, remaining games

Weaker but still objective: does the corrected move match the game's required tag format?
Since the F damage manifested as parsing failures (failed→aborted migration), a correction that
itself breaks format would be direct evidence of content-side harm. Semantic quality of these
games' corrections is left to Tier 4.

In [6]:
FORMATS = {
    "adventuregame":    re.compile(r"^\s*>\s*\S+"),
    "privateshared":    re.compile(r"^\s*(ANSWER|ASIDE):", re.I),
    "guesswhat":        re.compile(r"^\s*(QUESTION|GUESS|ANSWER):", re.I),
    "taboo":            re.compile(r"^\s*(CLUE|GUESS):", re.I),
    "referencegame":    re.compile(r"(Expression|Answer):", re.I),
    "imagegame":        re.compile(r"(Command:|DONE)", re.I),
    "matchit_ascii":    re.compile(r"(DESCRIPTION|QUESTION|ANSWER|DECISION):", re.I),
}
covered = {x["game"] for x in RESULTS}
for _, p in DF.iterrows():
    if p["game"] in covered or p["game"] not in FORMATS:
        continue
    rx = FORMATS[p["game"]]
    ok = bool(rx.search(p["chosen"].strip()))
    RESULTS.append(verdict_row(p, "valid" if ok else "invalid",
                               "format tag present" if ok else "missing required tag",
                               "T3-format"))
t3 = pd.DataFrame([x for x in RESULTS if x["tier"] == "T3-format"])
display(t3.groupby(["size", "pairset", "game"]).verdict.value_counts().unstack(fill_value=0))
uncovered = sorted(set(DF.game.unique()) - {x["game"] for x in RESULTS})
print("games with no validator (Tier 4 only):", uncovered)

verdict                     invalid  valid
size pairset game                         
2B   failed  adventuregame        0     35
             guesswhat            0     16
             imagegame            2     28
             matchit_ascii        0     14
             privateshared        0     45
             referencegame        0     29
             taboo                0      7
9B   failed  adventuregame        0     30
             guesswhat            0     27
             imagegame            1      6
             privateshared        0     45
             referencegame        0      9
             taboo                0      5

games with no validator (Tier 4 only): []


## Aggregate: correction validity vs the per-game damage map

The discriminating question: **does correction validity predict where F hurt?**
Per-game F damage (9B, clemscore delta vs SFT, from the results summary §2.4) is hardcoded
below. If validity is uniformly high — including in the games F destroyed — the damage
mechanism is distributional (off-policy chosen), not content quality.

In [7]:
R = pd.DataFrame(RESULTS)
ver = (R[R.verdict != "unverifiable"]
       .assign(valid=lambda d: d.verdict == "valid")
       .groupby(["size", "pairset", "game"])
       .agg(n=("valid", "size"), valid_pct=("valid", lambda s: 100 * s.mean())))
display(ver.round(1))

summary = (R.groupby(["size", "pairset"]).verdict
            .value_counts(normalize=True).unstack(fill_value=0) * 100)
print("\noverall verdict shares (%):")
display(summary.round(1))

# per-game F damage at 9B (clemscore delta vs SFT, val split) — results summary sec 2.4
F_DAMAGE_9B = {"adventuregame": -7, "codenames": -15, "guesswhat": +33, "imagegame": +11,
               "matchit_ascii": 0, "privateshared": +1, "referencegame": +20, "taboo": -19,
               "textmapworld": -41, "textmapworld_graphreasoning": -55,
               "textmapworld_specificroom": -67, "wordle": +10, "wordle_withclue": 0,
               "wordle_withcritic": 0}
fv = ver.reset_index().query("size == '9B' and pairset == 'failed'")
fv["f_damage"] = fv.game.map(F_DAMAGE_9B)
fv = fv.dropna(subset=["f_damage"])
if len(fv) >= 4:
    from scipy.stats import spearmanr
    rho, pval = spearmanr(fv.valid_pct, fv.f_damage)
    print(f"\nSpearman(correction validity, F per-game damage): rho={rho:.2f}, p={pval:.3f}  (n={len(fv)} games)")
display(fv[["game", "n", "valid_pct", "f_damage"]].sort_values("f_damage"))

n  valid_pct
size pairset game                                      
2B   aborted codenames                    50      100.0
             wordle                       16       81.2
             wordle_withclue               6       83.3
             wordle_withcritic             7       71.4
     failed  adventuregame                35      100.0
             codenames                    58      100.0
             guesswhat                    16      100.0
             imagegame                    30       93.3
             matchit_ascii                14      100.0
             privateshared                45      100.0
             referencegame                29      100.0
             taboo                         7      100.0
             textmapworld                 45       97.8
             textmapworld_graphreasoning  27      100.0
             textmapworld_specificroom    14      100.0
             wordle                        2      100.0
             wordle_withclue               1      100.0
             wordle_withcritic             2       50.0
9B   aborted codenames                    24      100.0
             wordle                        9       44.4
             wordle_withclue               6       83.3
             wordle_withcritic             3      100.0
     failed  adventuregame                30      100.0
             codenames                    69      100.0
             guesswhat                    27      100.0
             imagegame                     7       85.7
             privateshared                45      100.0
             referencegame                 9      100.0
             taboo                         5      100.0
             textmapworld                 45      100.0
             textmapworld_graphreasoning  27      100.0
             textmapworld_specificroom    14      100.0
             wordle                        2      100.0
             wordle_withclue               2      100.0
             wordle_withcritic             3       66.7


overall verdict shares (%):


verdict       invalid  unverifiable  valid
size pairset                              
2B   aborted      7.6           0.0   92.4
     failed       1.2           0.0   98.8
9B   aborted     14.3           0.0   85.7
     failed       0.7           0.3   99.0


Spearman(correction validity, F per-game damage): rho=-0.24, p=0.429  (n=13 games)


,game,n,valid_pct,f_damage
31,textmapworld_specificroom,14,100.000000,-67
30,textmapworld_graphreasoning,27,100.000000,-55
29,textmapworld,45,100.000000,-41
28,taboo,5,100.000000,-19
23,codenames,69,100.000000,-15
22,adventuregame,30,100.000000,-7
33,wordle_withclue,2,100.000000,0
34,wordle_withcritic,3,66.666667,0
26,privateshared,45,100.000000,1
32,wordle,2,100.000000,10


## Reading the results & Tier 4

- **High validity in the destroyed games** (textmapworld family legality, wordle consistency)
  + **no correlation between validity and damage** → the corrosion is *distributional*: the
  judge's moves were fine, but they are text the policy doesn't generate, and pushing
  probability toward them degraded parsing discipline. This closes the "GPT's moves were just
  bad" objection.
- **AB vs F validity contrast**: if similar, content quality cannot explain why AB helped and
  F hurt — the difference is what the pairs *target* (format the model half-knows vs strategy
  it doesn't), supporting the elicitation reading.
- **Tier 4 (not run)**: the semantic remainder (guesswhat question quality, taboo clue quality,
  imagegame instructions) needs a second LLM judge from a different family; agreement-rate on a
  ~50-pair sample is enough. The `RESULTS` dataframe marks exactly which pairs need it.

**Caveats.** Validators parse the pair's own prompt; `unverifiable` rows are reported above —
check their share before quoting any rate. Format regexes approximate each game's parser (the
GM's real parser is authoritative). The wordle sample is small (8 pairs at 9B). Legality ≠
optimality: a legal textmapworld move can still be strategically poor — Tier 2 bounds *content
harm*, it does not certify brilliance.